# Codification 1 : corrections manuelles d'avril à août

Deuxième étape de l'essai de correction de la codification 1 par machine learning (journal : [`tableau_resultats.md`](tableau_resultats.md)). Essai **bonus et indépendant** des autres branches de machine learning du projet.

**Objet** : trancher les codifications 1 d'avril à août (`PAY_2` à `PAY_6`, 34 valeurs chez 28 clients). Ces 28 clients ont tous aussi `PAY_1` = 1 : ils ne sont ni dans le train ni dans le test du machine learning ([`01_ml_septembre.ipynb`](01_ml_septembre.ipynb)), que ces corrections n'affectent donc pas. Elles seront appliquées **après** le machine learning, en correctif, s'il est concluant. Les `PAY_1` = 1 ne sont pas tranchés ici.

**Données** : niveau 0 du nettoyage (`data/creditcard_pret_ingestion.csv`, 30 000 clients, `EDUCATION` et `MARRIAGE` harmonisés, toutes les codifications 1 d'origine intactes). **`dpnm` est retiré dès le chargement** : il n'est utilisé à aucun moment de l'essai.

## 1. Données du niveau 0

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

# Racine du projet : premier dossier parent qui contient 'data'
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
df = df.drop(columns="dpnm")   # jamais utilisé dans ce notebook
print(f"Clients du niveau 0 : {len(df)} ; colonnes : {df.shape[1]} (dpnm retirée)")

PAY = [f"PAY_{n}" for n in range(1, 7)]
MOIS = {1: "septembre", 2: "août", 3: "juillet", 4: "juin", 5: "mai", 6: "avril"}

effectifs = pd.DataFrame({
    "Mois": [MOIS[n] for n in range(1, 7)],
    "Codifications 1": [int((df[f"PAY_{n}"] == 1).sum()) for n in range(1, 7)],
}, index=PAY)
effectifs["Part des clients (%)"] = (effectifs["Codifications 1"] / len(df) * 100).round(2)
display(effectifs)

**Comment lire** : une ligne par mois. Presque toutes les codifications 1 sont en septembre (`PAY_1`) ; les autres mois n'en ont que quelques dizaines au plus, traitées à la main dans la section 2.

## 2. Les codifications 1 des mois précédents (`PAY_2` à `PAY_6`)

Les clients ci-dessous ont au moins une codification 1 entre avril et août. Chaque client est affiché **en entier**, sur une ligne, avec toutes les colonnes du dataset sauf `dpnm` ; les codifications 1 sont surlignées.

**Ordre des colonnes** : d'abord le profil (`LIMIT_BAL`, `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`) et `BILL_AMT1` (solde de fin septembre, exigible en octobre) ; puis, pour chaque mois de septembre (n = 1) à avril (n = 6), les trois colonnes qui vont ensemble : **`PAY_n`** (codification du mois), **`PAY_AMTn`** (paiement du mois), **`BILL_AMT(n+1)`** (facture exigible ce mois-ci, que ce paiement règle). Avril n'a pas de facture exigible visible (`BILL_AMT7` n'existe pas).

Les clients sont groupés selon le mois de leur **codification 1 la plus ancienne**, une cellule par groupe : d'abord juin (`PAY_4`), puis juillet (`PAY_3`), puis août (`PAY_2`). On corrige dans cet ordre, du plus ancien au plus récent : une codification corrigée sert de point d'appui pour le mois suivant.

Un paiement de régularisation peut être enregistré avec un mois de décalage : regarder `PAY_AMT(n-1)` avant de conclure à une absence de paiement.

In [ ]:
# Colonnes groupées par mois : PAY_n, PAY_AMTn, BILL_AMT(n+1), de septembre (n = 1) à avril (n = 6)
COLONNES_CLIENT = ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "BILL_AMT1"]
for n in range(1, 7):
    COLONNES_CLIENT += [f"PAY_{n}", f"PAY_AMT{n}"] + ([f"BILL_AMT{n + 1}"] if n < 6 else [])
assert sorted(COLONNES_CLIENT) == sorted(df.columns)   # toutes les colonnes, sauf dpnm (retirée au chargement)

def plus_ancien_1(client):
    # Mois de la codification 1 la plus ancienne entre PAY_2 et PAY_6 (n le plus grand)
    return max(n for n in range(2, 7) if client[f"PAY_{n}"] == 1)

def surligner_1(ligne):
    return ["background-color: #ffe08a; font-weight: bold" if c in PAY and v == 1 else "" for c, v in ligne.items()]

def afficher_clients(table, titre):
    display(Markdown(titre))
    display(table[COLONNES_CLIENT].set_index("ID").style.apply(surligner_1, axis=1)
            .format("{:,.0f}", subset=[c for c in COLONNES_CLIENT if c.startswith(("BILL_AMT", "PAY_AMT", "LIMIT_BAL"))], thousands=" "))

def afficher_groupe(n):
    groupe = clients[clients["plus_ancien_1"] == n].sort_values("ID")
    afficher_clients(groupe, f"**Codification 1 la plus ancienne en {MOIS[n]} (`PAY_{n}`) : {len(groupe)} client(s)**")

clients = df[(df[PAY[1:]] == 1).any(axis=1)].copy()
clients["plus_ancien_1"] = clients.apply(plus_ancien_1, axis=1)
print(f"Clients avec au moins une codification 1 d'avril à août : {len(clients)}")
print(f"Codifications 1 concernées (PAY_2 à PAY_6) : {int((clients[PAY[1:]] == 1).sum().sum())}")
print(f"Parmi ces clients, codifiés 1 aussi en septembre : {int((clients['PAY_1'] == 1).sum())}")
print("Clients par mois de la codification 1 la plus ancienne :",
      {MOIS[n]: int(k) for n, k in clients["plus_ancien_1"].value_counts().sort_index(ascending=False).items()})

In [ ]:
afficher_groupe(4)   # juin (PAY_4)

In [ ]:
afficher_groupe(3)   # juillet (PAY_3)

**Août : règle avant la correction à la main.** Constatée sur plusieurs clients : beaucoup n'ont plus d'encours à la fin d'août. Si la codification de juillet est saine (`PAY_3` <= 0) et que le solde de fin août est nul ou créditeur (`BILL_AMT2` <= 0), le client a tout réglé : il n'a rien à rattraper et reprend sa codification de juillet (`PAY_2` = `PAY_3`). Les clients corrigés par la règle sont affichés après correction ; seuls les restants sont affichés ensuite, pour la correction à la main.

In [ ]:
# Règle d'août : si PAY_3 <= 0 et BILL_AMT2 <= 0, alors PAY_2 = PAY_3 (plus d'encours fin août, rien à rattraper)
regle_aout = (df["PAY_2"] == 1) & (df["PAY_3"] <= 0) & (df["BILL_AMT2"] <= 0)
df.loc[regle_aout, "PAY_2"] = df.loc[regle_aout, "PAY_3"]
ids_regle = set(df.loc[regle_aout, "ID"])
afficher_clients(df[regle_aout].sort_values("ID"), f"**Corrigés par la règle d'août, après correction : {len(ids_regle)} client(s)**")

clients = clients[~clients["ID"].isin(ids_regle)]   # il ne reste que les clients à corriger à la main

In [ ]:
afficher_groupe(2)   # août (PAY_2), clients restants après la règle

**Comment lire** : une ligne par client. Pour trancher la codification 1 du mois n, lire son trio : la facture exigible `BILL_AMT(n+1)` était-elle nulle (pas de retard possible), le paiement `PAY_AMTn` la couvre-t-il ? Puis les trios voisins : codification d'avant (à droite), et mois suivant (à gauche), pour un paiement enregistré avec un mois de décalage.

## 3. Décisions à la main

Une entrée par client corrigé : `ID: {n: nouvelle codification de PAY_n}`, pour n = 2 à 6, avec la raison métier en commentaire, tirée des montants. On peut réécrire toute la ligne de codifications d'un client jusqu'à `PAY_2` inclus. `PAY_1` n'est pas tranché ici : il relève du modèle de septembre.

Chaque client corrigé est affiché avant et après correction.

In [ ]:
# ID : {n : nouvelle valeur de PAY_n}, pour n = 2 à 6
corrections = {
    # Codification 1 la plus ancienne en juin (PAY_4)
    6783: {4: 0, 3: 0, 2: 0},    # paie chaque mois l'essentiel de sa facture : remis à 0 jusqu'à PAY_2
    # 11498 : les 2 d'avril et de mai ne sont pas choquants (la banque met jusqu'à deux mois à régulariser après le gros paiement d'avril) ;
    # en juin, quasi aucun paiement (PAY_AMT4 = 100 sur 285 901 dus), puis des paiements sous la mensualité minimale de 3 % (TYPE_USAGE, page 4.3) :
    # 2,6 % en juillet et en août, avec un solde au-dessus du plafond (sans savoir si ce plafond était déjà celui appliqué avant) : retard maintenu
    11498: {4: 2, 3: 2, 2: 2},
    # Codification 1 la plus ancienne en juillet (PAY_3)
    # 892 : PAY_4 = 2 laissé tel quel (posé alors qu'il paie 38 % de sa facture : semble prématuré, comme une information prédictive de la banque) ;
    # juillet, quasi aucun paiement (PAY_AMT3 = 3 sur 12 599 dus) : retard ; août, il repaie (5 014 sur 15 882, 32 %) : retour à 0
    892: {3: 2, 2: 0},
    # 1967 : juillet, 0 (il a soldé sa facture en juin, et paie 770 sur 2 442 dus en juillet) ;
    # août, montants incohérents : le solde baisse de 73 960 à 43 480 alors qu'il ne rembourse que 1 000 ; malgré ce PAY_AMT2 faible,
    # l'encours a bien baissé : 0
    1967: {3: 0, 2: 0},
    # Codification 1 en août (PAY_2), clients restants après la règle d'août
    5645: {2: 0},    # sain d'avril à juillet ; août, paie 177 080 sur 177 741 dus (99,6 %) : facture réglée, 0
    19195: {2: 2},   # en retard depuis avril ; août, paie 74 sur 368 318 dus, solde au-dessus du plafond fin août : retard maintenu
    25176: {2: 2},   # en retard depuis avril ; août, ne paie rien sur 49 555 dus, solde au-dessus du plafond fin août : retard maintenu
}

for id_client, valeurs in corrections.items():
    assert id_client in set(clients["ID"]), f"client {id_client} : pas de codification 1 d'avril à août"
    assert all(2 <= n <= 6 for n in valeurs), f"client {id_client} : seules PAY_2 à PAY_6 se corrigent ici"
    ligne = df["ID"] == id_client
    avant = df.loc[ligne].copy()
    for n, valeur in valeurs.items():
        df.loc[ligne, f"PAY_{n}"] = valeur
    afficher_clients(pd.concat([avant, df.loc[ligne]]).assign(ID=lambda t: [f"{id_client} avant", f"{id_client} après"]),
                     f"**Client {id_client}**")

restants = int((df[PAY[1:]] == 1).sum().sum())
print(f"Clients corrigés : {len(corrections)} ; codifications 1 restantes d'avril à août : {restants}")